# Notebook 05: AI Agent Demo

**Mục tiêu học tập:**
- Hiểu cách AI Agent tương tác với MCP Tool Server
- Xem agent tự động tìm cấu hình MPC tốt
- Hiểu vai trò của safety validation

**Kiến thức nền:**
- [MCP Documentation](https://modelcontextprotocol.io/)
- [Hugging Face Agents Course](https://huggingface.co/learn/agents-course)

In [ ]:
import sys
sys.path.insert(0, '..')

from mcp_server.server import MCPServer
from mcp_server.safety import get_safe_config_range
from agent.agent_loop import AgentLoop

## 1. MCP Tool Server

Tool server là cầu nối giữa Agent và hệ thống MPC.

In [ ]:
server = MCPServer()

print("Available tools:")
for tool in server.list_tools():
    print(f"  - {tool['name']}: {tool['description']}")

## 2. Safety Validation

Mọi config phải vượt qua validation trước khi chạy.

In [ ]:
# Valid config
valid_config = {
    'dt': 0.1, 'horizon': 10,
    'Q': [1.0, 1.0, 0.5, 0.1], 'R': [0.1, 0.1],
    'max_steer': 0.5, 'max_accel': 2.0,
    'max_speed': 3.0, 'vehicle_length': 0.3, 'num_iterations': 3,
}
ok, reason = server.call('validate_config', config=valid_config)
print(f"Valid config: ok={ok}, reason='{reason}'")

# Invalid config (negative Q)
bad_config = dict(valid_config)
bad_config['Q'] = [1.0, -1.0, 0.5, 0.1]
ok, reason = server.call('validate_config', config=bad_config)
print(f"Bad config: ok={ok}, reason='{reason}'")

# Safe ranges
print("\nSafe parameter ranges:")
for param, (lo, hi) in get_safe_config_range().items():
    print(f"  {param}: [{lo}, {hi}]")

## 3. Run Agent

Agent tự động thử nhiều cấu hình MPC và tìm config tốt nhất.

In [ ]:
agent = AgentLoop(server=server, max_trials=5)

results = agent.run(
    goal={'rmse_target': 0.3, 'collision_target': 0, 'max_trials': 5},
    scenario_name='basic_circle',
)

In [ ]:
# Show results
print(f"\nBest config: {results['best_config']}")
print(f"Best metrics: {results['best_metrics']}")
print(f"Total trials: {results['trials']}")

# History table
if results['history']:
    print("\nTrial history:")
    for h in results['history']:
        rmse = h.get('metrics', {}).get('position_rmse', 'N/A')
        coll = h.get('metrics', {}).get('collision_count', 'N/A')
        print(f"  Trial {h['trial']}: status={h['status']}, RMSE={rmse}, collisions={coll}")

## 4. Agent trên map có vật cản

In [ ]:
agent2 = AgentLoop(server=server, max_trials=5)

results2 = agent2.run(
    goal={'rmse_target': 0.5, 'collision_target': 0, 'max_trials': 5},
    scenario_name='obstacles',
)

print(f"\nBest metrics: {results2['best_metrics']}")

## Tóm tắt

- **MCP Tool Server** cung cấp interface chuẩn để agent gọi simulation
- **Safety validation** đảm bảo agent không thể dùng config nguy hiểm
- **Agent Loop** tự động thử nghiệm, học từ kết quả, và đề xuất config tốt hơn
- Agent hiện dùng rule-based strategy; có thể mở rộng bằng LLM

**Bài tập:**
1. Thêm tool mới vào MCP server (ví dụ: `get_scenario_info`)
2. Thử goal khác nhau (ưu tiên smoothness thay vì RMSE)
3. Viết strategy mới trong `agent/prompts.py`